# 🧹 Data Cleaning — Sales Intelligence Dataset

**Project:** Interactive Sales Intelligence Dashboard  
**Author:** rihan-png  
**Dataset columns:** `date`, `region`, `sales`, `conversions`, `visits`

---
This notebook walks through the complete data cleaning pipeline used to prepare `cleaned_data.csv` for the interactive dashboard.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported ✅')

## 2. Load Raw Data

In [ ]:
df = pd.read_csv('../src/cleaned_data.csv')
print(f'Shape: {df.shape}')
df.head(10)

## 3. Inspect Data Types & Missing Values

In [ ]:
print('=== Data Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Descriptive Stats ===')
df.describe()

## 4. Handle Missing Values

In [ ]:
missing_before = df.isnull().sum().sum()

# Drop rows where critical fields are null
df = df.dropna(subset=['date', 'region', 'sales'])

# Fill numeric nulls with column median
for col in ['conversions', 'visits']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

missing_after = df.isnull().sum().sum()
print(f'Missing values: {missing_before} → {missing_after} ✅')

## 5. Remove Duplicates

In [ ]:
dups_before = len(df)
df = df.drop_duplicates()
dups_after = len(df)
print(f'Rows before: {dups_before} | After dedup: {dups_after} | Removed: {dups_before - dups_after} ✅')

## 6. Type Casting & Normalisation

In [ ]:
# Parse dates
df['date'] = pd.to_datetime(df['date'])

# Ensure numeric types
df['sales']       = pd.to_numeric(df['sales'],       errors='coerce')
df['conversions'] = pd.to_numeric(df['conversions'], errors='coerce')
df['visits']      = pd.to_numeric(df['visits'],      errors='coerce')

# Strip whitespace from region
df['region'] = df['region'].str.strip().str.title()

print('Types after casting:')
print(df.dtypes)

## 7. Outlier Detection & Treatment

In [ ]:
# IQR method for sales outliers
Q1 = df['sales'].quantile(0.25)
Q3 = df['sales'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['sales'] < lower) | (df['sales'] > upper)]
print(f'Outliers detected: {len(outliers)}')

# Cap (winsorise) instead of dropping
df['sales'] = df['sales'].clip(lower=lower, upper=upper)
print(f'Sales range after treatment: {df["sales"].min():.0f} – {df["sales"].max():.0f} ✅')

## 8. Feature Engineering

In [ ]:
df['conversion_rate'] = (df['conversions'] / df['visits']).round(4)
df['month']           = df['date'].dt.to_period('M').astype(str)
df['day_of_week']     = df['date'].dt.day_name()

print('New features added:')
df[['date', 'region', 'sales', 'conversion_rate', 'month', 'day_of_week']].head(8)

## 9. Exploratory Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Sales Intelligence — EDA Snapshots', fontsize=14, fontweight='bold')

# Sales by region
region_sales = df.groupby('region')['sales'].sum().sort_values(ascending=False)
axes[0].bar(region_sales.index, region_sales.values,
            color=['#2451a6', '#10b981', '#f59e0b', '#7c3aed'])
axes[0].set_title('Total Sales by Region')
axes[0].set_ylabel('Sales (₹)')

# Sales trend
monthly = df.groupby('month')['sales'].sum()
axes[1].plot(monthly.index, monthly.values, marker='o', color='#2451a6', linewidth=2)
axes[1].set_title('Monthly Sales Trend')
axes[1].set_ylabel('Sales (₹)')
axes[1].tick_params(axis='x', rotation=45)

# Conversion rate distribution
axes[2].hist(df['conversion_rate'], bins=20, color='#10b981', edgecolor='white')
axes[2].set_title('Conversion Rate Distribution')
axes[2].set_xlabel('Conversion Rate')

plt.tight_layout()
plt.savefig('../src/assets/screenshots/eda_snapshot.png', dpi=100, bbox_inches='tight')
plt.show()
print('EDA saved ✅')

## 10. Key Insights

In [ ]:
print('=== KEY BUSINESS INSIGHTS ===')
print(f"Total Revenue      : ₹{df['sales'].sum():,.0f}")
print(f"Avg Conversion Rate: {df['conversion_rate'].mean()*100:.2f}%")
print(f"Best Region        : {region_sales.idxmax()} (₹{region_sales.max():,.0f})")
print(f"Best Month         : {monthly.idxmax()} (₹{monthly.max():,.0f})")
print(f"Total Records      : {len(df)}")

## 11. Export Cleaned Data

Save the base version (without the extra engineered columns) back to `src/cleaned_data.csv` for the dashboard.

In [ ]:
export_cols = ['date', 'region', 'sales', 'conversions', 'visits']
df_export = df[export_cols].copy()
df_export['date'] = df_export['date'].dt.strftime('%Y-%m-%d')
df_export.to_csv('../src/cleaned_data.csv', index=False)
print(f'Exported {len(df_export)} rows to ../src/cleaned_data.csv ✅')